# Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os 
import sys
sys.path.append("../src")

import matplotlib.pyplot as plot
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

import plotly.io as pio
import plotly.express as px

import optuna
import pandas as pd
import numpy as np
import scipy
import sklearn as sk
import ruptures as rpt
import pymannkendall as mk
from collections import Counter, defaultdict
from tqdm import tqdm
import statsmodels.api as sm
from itertools import combinations
from bs4 import BeautifulSoup
import re
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix
from mlxtend.frequent_patterns import apriori, association_rules
from prefixspan import PrefixSpan
from functools import reduce

from bisect import bisect_left,bisect_right
from global_utils.graphs_utils import get_subplots, prepare_subplots, fancy_histogram, color_dic, arrange_twin_plots
from global_utils.files_utils import create_tree_folder
from ticket_analysis.ticket_utils import *
import dask.dataframe as dd
import duckdb
import subprocess
import pickle
pd.set_option("display.max_columns", None)
import warnings
warnings.filterwarnings("ignore")# warnings.filterwarnings("default")


# Read Files

In [ ]:
DATA_DIR = "../Database"
fontsize = 18
if os.path.exists(DATA_DIR):
    print("Can see Data_Dir")
colors = color_dic["default"]

In [ ]:
meter_values = [pd.read_csv(os.path.join(DATA_DIR, 'MeterValues.csv')), pd.read_csv(os.path.join(DATA_DIR, "MeterValues_Extra.csv")), pd.read_csv(os.path.join(DATA_DIR, "MeterValues_Extra.csv"))]
meter_values = pd.concat(meter_values, ignore_index=True)
meter_values['MeterValueTimeStamp'] = pd.to_datetime(meter_values['MeterValueTimeStamp'], format="mixed")
meter_values['UpdatedTimeStamp'] = pd.to_datetime(meter_values['UpdatedTimeStamp'], format="mixed")


In [ ]:
transactions = [pd.read_csv(os.path.join(DATA_DIR, "Transactions.csv")),pd.read_csv(os.path.join(DATA_DIR, "Transactions_Extra.csv")), pd.read_csv(os.path.join(DATA_DIR, "LegacyTransactions_new.csv"))]
transactions = pd.concat(transactions, ignore_index=True)
transactions["StartAt"] = pd.to_datetime(transactions["StartAt"], format="mixed")
transactions["EndAt"] = pd.to_datetime(transactions["EndAt"], format="mixed")

In [ ]:
events = pd.read_parquet(os.path.join(DATA_DIR, "Events_Trimmed.parquet"))

In [ ]:
name_of_chargers = list(set(transactions["ChargerID"].unique()) & set(events["device_id"].unique()) & set(meter_values["ChargerID"].unique()))

In [ ]:
charger_transactions = {key: value.sort_values(by="StartAt") for key, value in transactions.groupby("ChargerID")}
charger_events = {key: value.sort_values(by="event_time") for key, value in events.groupby("device_id", observed=True)}
charger_meter_values = {key: {trId: group.sort_values(by="MeterValueTimeStamp") for trId, group in value.groupby("TransactionID")} for key, value in meter_values.groupby("ChargerID")}


In [ ]:
merge_isp = pd.read_csv(os.path.join(DATA_DIR, "Merged_ISP.csv"))
merge_isp["Date"] = pd.to_datetime(merge_isp["Date"]).dt.tz_localize("UTC")


# Function Definition

In [ ]:
def get_meter_values_dic(meter_values_df):
    times = meter_values_df["MeterValueTimeStamp"].to_list()
    json_dic = [ { key: float(val) for key, val in json.loads(raw_data).items() } for raw_data in meter_values_df["RawData"]  ]
    keys = set([key for subdic in json_dic for key in subdic.keys()])
    values_dic = {key:[val.get(key, 0) for val in json_dic] for key in keys}
    return times, values_dic


def get_slopes_energies(charger):


    linear = lambda x, a, b: x*a+b
    slopes_per_charger = []
    for index, (_, transaction) in enumerate(charger_transactions[charger].iterrows()):
        StartAt = transaction["StartAt"]
        EndAt = transaction["EndAt"]
        EnergyDelivered = transaction["EnergyDelivered"]
        if np.isnan(EnergyDelivered) or EnergyDelivered < 0:
            EnergyDelivered = 0
        
        transactions_duration = (EndAt-StartAt).total_seconds()
        specific_meter_values_df = charger_meter_values.get(charger, {}).get(transaction["ChargerTransactionID"], pd.DataFrame())

        coef = [None, None]
        if len(specific_meter_values_df) > 2:
            xs, dic = get_meter_values_dic(specific_meter_values_df)
            enes = np.array(dic["EnergyDelivered"])
            enes = enes - enes[0]
            EnergyDelivered = max(enes)

            start_instance = bisect_right(enes, 0) -1
            start_intsance = min(start_instance, len(enes)-1)

            stop_instance = bisect_left(enes, EnergyDelivered*0.95)+1
            stop_instance = min(stop_instance, len(enes)-1)

            start_instance, stop_instance = sorted([start_instance, stop_instance])
            StartAt, EndAt = xs[start_instance], xs[stop_instance]

            xs = np.array([(element-xs[0]).total_seconds() for element in xs])
            xs = xs[start_instance:stop_instance-1]
            enes = enes[start_instance:stop_instance-1]
            if len(xs) > 2:
                coef, coef_cov = scipy.optimize.curve_fit(linear, xs/3600, enes/1000)

        slopes_per_charger.append( [coef[0], (EnergyDelivered)/((EndAt-StartAt).total_seconds()+1)*3600/1000, StartAt] )

    return slopes_per_charger

def get_slopes_from_metering(specific_meter_values_df, EnergyDelivered, StartAt,EndAt):

    linear = lambda x, a, b: x*a+b
    coef = [None, None]
    if len(specific_meter_values_df) > 2:
        xs, dic = get_meter_values_dic(specific_meter_values_df)
        enes = np.array(dic["EnergyDelivered"])
        enes = enes - enes[0]
        EnergyDelivered = max(enes)

        start_instance = bisect_right(enes, 0) -1
        start_intsance = min(start_instance, len(enes)-1)

        stop_instance = bisect_left(enes, EnergyDelivered*0.95)+1
        stop_instance = min(stop_instance, len(enes)-1)

        start_instance, stop_instance = sorted([start_instance, stop_instance])
        StartAt, EndAt = xs[start_instance], xs[stop_instance]

        xs = np.array([(element-xs[0]).total_seconds() for element in xs])
        xs = xs[start_instance:stop_instance-1]
        enes = enes[start_instance:stop_instance-1]
        if len(xs) > 2:
            coef, coef_cov = scipy.optimize.curve_fit(linear, xs/3600, enes/1000)


    return  [coef[0], (EnergyDelivered)/((EndAt-StartAt).total_seconds()+1)*3600/1000, StartAt] 


In [ ]:
def get_data_gaps_df(transactions_df, threshold_gap=pd.Timedelta("5D")):
    transactions_df["Next StartAt"] = transactions_df["StartAt"].shift(-1)
    transactions_df["Gap"] = (transactions_df["Next StartAt"] - transactions_df["EndAt"])
    data_gaps = transactions_df[transactions_df["Gap"]>threshold_gap]
    return data_gaps

def merge_intervals(periods):
    if not periods:
        return []

    intervals = sorted(
        (min(st,ed), max(st,ed))
        for st, ed in periods
    )

    merged = [list(intervals[0])]
    for st, ed in intervals[1:]:
        last_st, last_ed = merged[-1]
        if st < last_ed or st == last_ed:
            merged[-1][1] = max(last_ed, ed)
        else:
            merged.append([st,ed])
    return merged

def intersect_intervals(chA, chB):
    result = []
    i, j = 0, 0
    while i < len(chA) and j < len(chB):
        lo = max([chA[i][0], chB[j][0]])
        hi = min([chA[i][1], chB[j][1]])
        if lo < hi:
            result.append([lo, hi])
        # Move forward whichever interval ends first
        if chA[i][1] < chB[j][1]:
            i += 1
        else:
            j += 1
    return result


In [ ]:
def closest_previous_or_nearest(row):
    events = row["event_time"]
    date = row["Date"]

    if not events or pd.isna(date):
        return pd.NaT

    events = [e for e in events if pd.notna(e)]

    if not events:
        return pd.NaT

    prev_deltas = [e - date for e in events if e < date]

    if prev_deltas:
        return max(prev_deltas)

    all_deltas = [e - date for e in events]
    return min(all_deltas, key=abs)

# ISP Tickets

In [ ]:
sorted_merge_isp = merge_isp.groupby("charging_station_id").agg(count=( "charging_station_id","size" ), Date=("Date", list)).sort_values("count", ascending=0)
sorted_merge_isp_only_events = merge_isp_only_events.groupby("charging_station_id").agg(count=( "charging_station_id","size" ), Date=("Date", list)).sort_values("count", ascending=0)
sorted_merge_isp_only_transactions = merge_isp_only_transactions.groupby("charging_station_id").agg(count=( "charging_station_id","size" ), Date=("Date", list)).sort_values("count", ascending=0)

In [ ]:
fig, ax = get_subplots()
fig2, ax2 = get_subplots()
fig3, ax3 = get_subplots(1,2,figsize=(15,5))
counts_e = sorted_merge_isp_only_events["count"]
counts_t = sorted_merge_isp_only_transactions["count"]

bins=np.arange(0, 21, 1)
ax.hist(counts_e, bins=bins, weights=np.ones(len(counts_e))/len(counts_e)*100, edgecolor="k")
print(np.mean(counts_e))
print(np.mean(counts_t))
ax2.hist(counts_t, bins=bins, weights=np.ones(len(counts_t))/len(counts_t)*100, edgecolor="k")
ax.set_ylim(0,50)
ax.set_title("Ticket Count Distribution per charger", fontsize=18)
ax.set_ylabel("Chargers (%)", fontsize=18)
ax.set_xlabel("Number of tickets", fontsize=18)

ax3[0].hist(counts_e, bins=bins, weights=np.ones(len(counts_e))/len(counts_e)*100, edgecolor="k")
ax3[1].hist(counts_t, bins=bins, weights=np.ones(len(counts_t))/len(counts_t)*100, edgecolor="k")
for a in ax3:
    a.set_ylabel("Chargers (%)", fontsize=18)
    a.set_xlabel("Number of tickets", fontsize=18)
    a.set_ylim(0,70)
    a.xaxis.set_major_locator(ticker.MultipleLocator(5))

fig3.suptitle("Ticket count distribution per charger, considering time range from: ", fontsize=18, y=1.02)

ax3[0].set_title("Events", fontsize=18)
ax3[1].set_title("Transactions", fontsize=18)


plot.show()

In [ ]:
ins = set(name_of_chargers).intersection(merge_isp["charging_station_id"].unique())
in_gaps = (set(ins) & set(all_gaps["ChargerID"].unique().tolist()))

chargers_in_gaps_and_tickets = transactions[transactions["ChargerID"].isin(ins)]
chargers_in_gaps_and_tickets = chargers_in_gaps_and_tickets.groupby("ChargerID").size()
chargers_in_gaps_and_tickets = chargers_in_gaps_and_tickets.sort_values(ascending=False).index.tolist()

counts = sorted_merge_isp[sorted_merge_isp.index.isin(name_of_chargers)]["count"] 

In [ ]:
fig, ax = get_subplots()
bins = np.arange(0,40,1)
ax.set_xlabel("Number of Tickets", fontsize=18)
ax.set_ylabel("Number of chargers", fontsize=18)
ax.set_title("Distribution of tickets per charger", fontsize=18)
ax.hist(counts, bins=bins)
plot.show()


In [ ]:
counts_transactions = []
counts_events = []
for specific_charger in name_of_chargers:
    counts = []
    specific_transactions = charger_transactions[specific_charger]
    min_time_transactions = specific_transactions["StartAt"].min()
    max_time_transactions = specific_transactions["EndAt"].max()

    specific_events = charger_events[specific_charger]
    min_time_events = specific_events["event_time"].min()
    max_time_events = specific_events["event_time"].max()
    try:
        dates = sorted_merge_isp.loc[specific_charger]["Date"]
        only_in_transactions = [d for d in dates if min_time_transactions<=d<max_time_transactions]
        only_in_events = [d for d in dates if min_time_events<=d<max_time_events]
        
        counts_transactions.append(len(only_in_transactions))
        counts_events.append(len(only_in_events))
    except KeyError:
        counts_transactions.append(0)
        counts_events.append(0)
    
counts_transactions = np.array(counts_transactions)
counts_events = np.array(counts_events)

In [ ]:
fig, ax = get_subplots(1,2, figsize=(15,6))
ax[0].hist(counts_transactions, bins=np.arange(0,10,1), edgecolor="k", weights=np.ones(len(counts_transactions))/len(counts_transactions)*100)
ax[1].hist(counts_events, bins=np.arange(0,25,1), edgecolor="k", weights=np.ones(len(counts_events))/len(counts_events)*100)
for a in ax:
    a.set_ylabel("Number of chargers", fontsize=18)
    a.set_xlabel("Number of tickets", fontsize=18)

ax[0].set_title("Distribution of tickets inside the transaction data", fontsize=15)
ax[1].set_title("Distribution of tickets inside the event data", fontsize=15)

print(" Mean transactions data: ", np.mean(counts_transactions))
print(" Mean events data: ", np.mean(counts_events))


plot.show()


In [ ]:
ticket_in_gap = []
min_distance_to_gap = []

total_tickets = 0
for specific_charger in ins:
    specific_gaps = all_gaps[all_gaps["ChargerID"]==specific_charger]
    specific_ticket = merge_isp[merge_isp["charging_station_id"]==specific_charger]
    date_min, date_max = pd.Timestamp(charger_transactions[specific_charger]["StartAt"].min()), pd.Timestamp(charger_transactions[specific_charger]["StartAt"].max())

    specific_ticket = specific_ticket[(specific_ticket["Date"] >= date_min) & (specific_ticket["Date"] <= date_max)]["Date"].tolist()
    total_tickets += len(specific_ticket)

    data_gaps = [(st, ed) for st,ed in specific_gaps[["StartAt", "Next StartAt"]].values.tolist()]

    for ticket_date in specific_ticket:
        in_gap = np.any([st< ticket_date < ed for st, ed in data_gaps])
        if len(data_gaps):
            if not in_gap:
                min_distance_to_gap.append(min([min([ticket_date-st, ticket_date-ed], key=lambda x:abs(x)) for st,ed in data_gaps] ,key=lambda x:abs(x)).total_seconds()/(3600))
            else:
                min_distance_to_gap.append(0)
        else:
            min_distance_to_gap.append(None)
        ticket_in_gap.append(bool(in_gap))

In [ ]:
fig, ax = plot.subplots()
ax.hist(min_distance_to_gap, bins=np.arange(-20,20,1))
ax.set_xlabel("hours")


In [ ]:
fig, ax = get_subplots()

for st, end in specific_gaps[["StartAt","Next StartAt"]].values.tolist():
    ax.fill_between([st,end],0,10, color="r", alpha=0.4)

dates = specific_ticket["Date"]
print(len(dates))

ax.scatter(dates, [5]*len(dates))
print(dates)

ax.set_xlim(date_min, date_max)
# ax.set_xlim(pd.Timestamp("2026-02-01"),None)
ax.tick_params(axis="x", rotation=90)



## Ticket Prediction

### Rate of Charge

In [ ]:
specific_charger = name_of_chargers[0]
temp = list(charger_meter_values[specific_charger].values())[0]


In [ ]:
slopes_per_charger = {}
for charger in name_of_chargers:
    slopes_per_charger[charger] = get_slopes_energies(charger)


In [ ]:
specific_charger = name_of_chargers[1]
specific_charger = "mPNYK3"
specific_charger = "T23aFI"
total_time_bins = []
freq = "5D"
diff = 5
bins_energy = np.arange(0, 250+diff, diff)
last_end = None
for st, ed in usable_time_ranges[specific_charger]:
    if last_end:
        total_time_bins.extend(pd.date_range(start=last_end, end=st, freq=freq))
    total_time_bins.extend(pd.date_range(start=st, end=ed, freq=freq))
    last_end = ed
specific_slopes = slopes_per_charger[specific_charger]

hist2d = np.zeros((len(bins_energy)-1, len(total_time_bins)-1))
total_counts = []

for index, (t_min,t_max) in enumerate(zip(total_time_bins[:-1], total_time_bins[1:])):
    temp = np.array( [
        s[0] if not s[0] is None else s[1]
        for s in specific_slopes if t_min <= s[2] < t_max
    ])
    if (t_max-t_min) < pd.Timedelta("2D"):
        temp = []
    total_counts.append(len(temp))
    if len(temp):
        weights = np.ones(len(temp)) / len(temp) * 100
        hist2d[:,index] = np.histogram(temp, bins=bins_energy, weights=weights)[0]
    else:
        hist2d[:,index] = np.array([None]*(len(bins_energy)-1))


fig, ax = get_subplots(figsize=(10,5))
fig2, ax2 = get_subplots(figsize=(10,5))

mesh = ax.pcolormesh(total_time_bins, bins_energy, hist2d)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("Transactions (%)", fontsize=fontsize)
cbar.ax.tick_params(labelsize=14)

xlim = ax.get_xlim()
try:
    tickets = sorted_merge_isp.loc[specific_charger]["Date"]
    print(f"There are a total of {len(tickets)} tickets")
    ax.scatter(tickets, [np.mean(bins_energy)]*len(tickets), marker = "x" )
except KeyError:
    print("No tickets")



ax2.bar(total_time_bins[:-1], total_counts, width=np.diff(total_time_bins),edgecolor="k", align="edge")
for a in [ax,ax2]:
    a.tick_params(axis="x", rotation=90)

ax.set_title(f"Charger: {specific_charger}")
ax.set_xlim(xlim)
ax2.set_xlim(xlim)
plot.show()

In [ ]:
specific_charger = "T23aFI"
specific_transactions = charger_transactions[specific_charger]
print(specific_transactions["EvseID"].unique())
starts = np.array(specific_transactions["StartAt"].to_list())

min_time = starts.min()
max_time = starts.max()

specific_events = charger_events[specific_charger]
specific_events = specific_events[(specific_events["name"]=="Connectivity Update") & (specific_events["event_time"] >= min_time) & (specific_events["event_time"] <= max_time)].sort_values("event_time")

prev_start = min_time
list_ = specific_events[["event_time", "value"]].values
prev_start = list_[0][0]
prev_value = list_[0][1]
updates = []
for date, value in list_[1:]:
    updates.append([prev_start, date, prev_value])
    prev_start = date
    prev_value = value
updates.append([prev_start, max_time, prev_value])



In [ ]:
transaction_id_bad = specific_transactions[(specific_transactions["StartAt"]>pd.Timestamp("2026-01-05").tz_localize("UTC")) & (specific_transactions["StartAt"]<pd.Timestamp("2026-01-09").tz_localize("UTC")) & (specific_transactions["EvseID"] == 1)]["ChargerTransactionID"].tolist()
bad_meter_values = [
    charger_meter_values[specific_charger].get(tra_id, pd.DataFrame())
    for tra_id in transaction_id_bad
]

transaction_id_good = specific_transactions[(specific_transactions["StartAt"]>pd.Timestamp("2026-03-01").tz_localize("UTC")) & (specific_transactions["StartAt"]<pd.Timestamp("2026-03-15").tz_localize("UTC")) & (specific_transactions["EvseID"]==1)]["ChargerTransactionID"].tolist()
good_meter_values = [
    charger_meter_values[specific_charger].get(tra_id, pd.DataFrame())
    for tra_id in transaction_id_good
]


In [ ]:
fig, ax = get_subplots()
nmr_of_bad_meter_values = []
last_time_meter_value_bad = []
for meter_val_df in bad_meter_values:
    time, dic = get_meter_values_dic(meter_val_df)
    # time = [(t - time[0]).total_seconds()/((time[-1]-time[0]).total_seconds()+1) for t in time ]
    time = [(t - time[0]).total_seconds()/(60) for t in time ]
    ene = np.array(dic["EnergyDelivered"]) - dic["EnergyDelivered"][0]
    nmr_of_bad_meter_values.append(len(time))
    last_time_meter_value_bad.append(time[-1])
    ax.plot(time, ene, color="r")

nmr_of_good_meter_values = []
last_time_meter_value_good = []
for meter_val_df in good_meter_values:
    time, dic = get_meter_values_dic(meter_val_df)
    # time = [(t - time[0]).total_seconds()/((time[-1]-time[0]).total_seconds()+1) for t in time ]
    time = [(t - time[0]).total_seconds()/(60) for t in time ]
    nmr_of_good_meter_values.append(len(time))
    last_time_meter_value_good.append(time[-1])
    ene = np.array(dic["EnergyDelivered"]) - dic["EnergyDelivered"][0]
    ax.plot(time, ene, color="b", alpha=0.1)

lines = [
    mlines.Line2D([],[], color="b", label="After Ticket", alpha=1),
    mlines.Line2D([],[], color="r", label="Right Before Ticket", alpha=1),
]
ax.legend(handles=lines)
ax.set_title("Comparisson between charging curves\n from two different periods", fontsize=18)
ax.set_xlabel("Time (min)", fontsize=18)
ax.set_ylabel("Energy Delivered (Wh)", fontsize=18)
ax.set_ylim(0, 0.8e5)
ax.set_xlim(-5, 120)
plot.show()



In [ ]:
specific_charger = "T23aFI"
specific_charger = "u6AQXQ"
specific_charger = "4HIbKg"
specific_charger = "LSurCD"
# specific_charger = "m5DaZZ"
# specific_charger = "mtSXKB"
# specific_charger = "nfeVUo"
# specific_charger = "vsLGPQ"
# specific_charger = "xtIIQT"
# specific_charger = "LSurCD"
specific_charger = "mtSXKB"
specific_charger = "0AQbZI"
specific_events = charger_events[specific_charger]
specific_transactions = charger_transactions[specific_charger]
specific_transactions["EnergyDelivered"].fillna(0, inplace=True)

text = "ConnectorID"
text = "EvseID"
evses  = specific_transactions.sort_values(text).groupby(text,sort=True)
evses = [ev[1][["StartAt", "EnergyDelivered"]] for ev in evses]


In [ ]:
time_bins = pd.date_range(start=specific_transactions["StartAt"].min().normalize(), end=specific_transactions["StartAt"].max().normalize() + pd.Timedelta("1D"))

updates = specific_events[specific_events["name"] == "connectivity update"][["event_time", "value"]].sort_values("event_time")
updates["next event"] = updates["event_time"].shift(-1)

temp = specific_events[specific_events["name"] == "Connector Availability Update"][["event_time", "value", "source"]]
temp = {(int(k.strip().split("/")[1]), int(k.strip().split("/")[3])) : val for k, val in temp.groupby("source")}
per_connector = {}
for k, val in temp.items():
    val["next_event_time"] = val["event_time"].shift(-1)
    val.iloc[-1,3] = val.iloc[-1]["event_time"]+pd.Timedelta(nanoseconds=1)
    per_connector[k] = val

# faulted = per_connector[(1,1)]
faulted = pd.concat([sublist for sublist in per_connector.values()])
faulted = faulted[~faulted["value"].isin(["Available", "Occupied"])][["event_time", "next_event_time"]]

disconnected = [(st,ed) for st,vl,ed in updates.values if vl == "Disconnected"]

In [ ]:

func = lambda x: sum(x)/(len(x)+1e-6)

counts = [[func( 
    ev[(st<=ev["StartAt"]) & (ev["StartAt"] < ed) & (ev["EnergyDelivered"]<1e6) ]["EnergyDelivered"].tolist()
) for st,ed in zip(time_bins[:-1], time_bins[1:])] for ev in evses]

bins_centers = [st + (ed-st)/2 for st, ed in zip(time_bins[:-1], time_bins[1:])]

In [ ]:
fig, ax = get_subplots()

for inx, ct in enumerate(counts):
    ax.plot(bins_centers, ct, label=inx+1)
    pass

leg = ax.legend(title="Outlet ID")
ax.add_artist(leg)
xlim = ax.get_xlim()

cross = [mlines.Line2D([],[],linestyle="None", marker="x", color="r",label="Ticket Submitted")]
leg = ax.legend(loc="upper right", handles=cross)
ax.add_artist(leg)


if counts:
    max_y = max([max(ct) for ct in counts])
for st, ed in disconnected:
    if ed-st < pd.Timedelta("1h"):
        continue
    ax.fill_between([st,ed], 0, max_y,color="r", alpha=0.1)

for st, ed in faulted.values:
    if ed-st < pd.Timedelta("10min"):
        continue
    ax.fill_between([st,ed], 0, max_y,color="b", alpha=0.1)

specific_tickets = merge_isp_only_events[merge_isp_only_events["charging_station_id"] == specific_charger]["Date"].tolist()

ax.scatter(specific_tickets, [max_y/2]*len(specific_tickets), marker="x", color="r")

# ax.set_xlim(xlim)

ax.set_xlim(pd.Timestamp("2025-11-01T00:00:00+00:00"), pd.Timestamp("2026-03-31T00:00:00+00:00"))
# ax.set_xlim(pd.Timestamp("2025-11-01T00:00:00+00:00"), pd.Timestamp("2025-11-30T00:00:00+00:00"))
# ax.set_xlim(pd.Timestamp("2025-09-01T00:00:00+00:00"), pd.Timestamp("2025-09-30T00:00:00+00:00"))

ax.tick_params(axis="x", rotation=90)
ax.set_ylabel("Energy Delivered (Wh)", fontsize=18)
ax.set_title(" Behaviour of energy delivered per connector ", fontsize=18)
plot.show()


### Payment Terminal

In [ ]:
payment_terminal = events[events["name"].str.contains("payment", case=False, na=False)]
total_counts = payment_terminal.shape[0]
payment_terminal = payment_terminal.groupby("device_id", observed=True, as_index=False).agg({"event_time":list})

payment_terminal["min"] = payment_terminal["event_time"].apply(min)
payment_terminal["max"] = payment_terminal["event_time"].apply(max)

tickets_terminal = merge_isp_only_events[merge_isp_only_events["Title"].str.contains(r"\b(payment|terminal|pay|card|rfidcase)\b", case=False, na=False)]

tickets_terminal = tickets_terminal[["charging_station_id", "Title", "Date"]]

In [ ]:
merged = payment_terminal.merge(
    tickets_terminal[["charging_station_id", "Date"]],
    left_on="device_id",
    right_on="charging_station_id",
    how = "outer"
)

merged["device_id"] = merged["device_id"].fillna(merged["charging_station_id"])
merged["charging_station_id"] = merged["charging_station_id"].fillna(merged["device_id"])

merged["event_time"] = merged["event_time"].apply(lambda x: x if isinstance(x, list) else [])


merged.drop(columns="charging_station_id", inplace=True)
merged.set_index("device_id", inplace=True)
mask = merged["Date"].notna() & (merged["min"] <= merged["Date"]) & ( merged["Date"] <= merged["max"]) 

merged["closest"] = merged.apply(closest_previous_or_nearest,
    axis = 1
)

s = merged[mask]["closest"]
num = merged[mask][merged["Date"].notna() & merged["event_time"].apply(lambda x: len(x)== 0)].shape[0]
event_without_ticket = merged[merged["Date"].isna() & merged["event_time"].apply(lambda x: len(x)!=0)]#.size
event_without_ticket = sum(event_without_ticket["event_time"].apply(lambda x: len(x)))

In [ ]:
display(s)
print(num)
print("events without tickets:", event_without_ticket, " total nmr events: ", total_counts, "| ", event_without_ticket/total_counts)


#### Specific Charger

In [ ]:
specific_charger = "vWbIej"
specific_tickets = tickets_terminal[tickets_terminal["charging_station_id"] == specific_charger]["Date"].tolist()
specific_events = payment_terminal[payment_terminal["device_id"] == specific_charger]
event_times = specific_events["event_time"].explode().tolist()

limits = specific_events[["min","max"]].values[0]

In [ ]:
fig, ax = get_subplots()

ax.scatter(event_times, [1]*len(event_times), marker = "." )
ax.scatter(specific_tickets, [1]*len(specific_tickets))
ax.axvline(limits[0])
ax.axvline(limits[1])
ax.tick_params(axis="x", rotation=90)


### Outlet

In [ ]:

outlet_events = events[events["name"].str.contains("outlet", case=False, na=False)]
total_counts = outlet_events.shape[0]

outlet_events = outlet_events.groupby("device_id", observed=True, as_index=False).agg({"event_time":list})

outlet_events["min"] = outlet_events["event_time"].apply(min)
outlet_events["max"] = outlet_events["event_time"].apply(max)

tickets_outlet = merge_isp_only_transactions[merge_isp_only_transactions["Title"].str.contains(r"(outlet|charging|connector|plug)", case=False, na=False)]
tickets_outlet = merge_isp_only_transactions
tickets_outlet = tickets_outlet[["charging_station_id", "Title", "Date"]]

In [ ]:
merged = outlet_events.merge(
    tickets_outlet[["charging_station_id", "Date"]],
    left_on="device_id",
    right_on="charging_station_id",
    how = "outer"
)

merged["device_id"] = merged["device_id"].fillna(merged["charging_station_id"])
merged["charging_station_id"] = merged["charging_station_id"].fillna(merged["device_id"])

merged["event_time"] = merged["event_time"].apply(lambda x: x if isinstance(x, list) else [])

merged.drop(columns="charging_station_id", inplace=True)
merged.set_index("device_id", inplace=True)
mask = merged["Date"].notna() & (merged["min"] <= merged["Date"]) & ( merged["Date"] <= merged["max"]) 


merged["closest"] = merged.apply(closest_previous_or_nearest,
    axis = 1
)

In [ ]:
s = merged[mask]["closest"]
num = merged[mask][merged["event_time"].apply(lambda x: len(x)== 0)].shape[0]
event_without_ticket = merged[merged["Date"].isna() & merged["event_time"].apply(lambda x: len(x)!=0)]#.size
event_without_ticket = sum(event_without_ticket["event_time"].apply(lambda x: len(x)))

In [ ]:
for cl in s.tolist():
    if cl > pd.Timedelta(seconds=0):
        print(cl)
print(num)
print("events without tickets:", event_without_ticket, " total nmr events: ", total_counts, "| ", event_without_ticket/total_counts)


In [ ]:
connector_status = events[events["name"] == "Connector Availability Update"][["device_id", "source", "event_time", "value"]]
connector_status = {
    name : {
        (int(source.split("/")[1]), int(source.split("/")[3])) : tab[["event_time", "value"]].sort_values("event_time").assign(next_event_time=lambda df:df["event_time"].shift(-1).fillna(df["event_time"])).loc[lambda d: ~d["value"].isin(["Occupied", "Available"]), ["event_time", "next_event_time"] ].values.tolist()
        for source, tab in val.groupby("source")
    }
    for name, val in connector_status.groupby("device_id")
}

charger_status = events[events["name"] == "Connectivity Update"][["device_id", "event_time", "value"]]
charger_status = {
    name : tab[["event_time", "value"]].sort_values("event_time").assign(next_event_time=lambda df: df["event_time"].shift(-1).fillna(df["event_time"])).loc[lambda d: d["value"].eq("Disconnected"), ["event_time", "next_event_time"]].values.tolist()
    for name, tab in charger_status.groupby("device_id")
}

faulted_periods = {
    name : pd.IntervalIndex.from_tuples([(st,ed) for st,ed in merge_intervals([period for periods_conn in vals.values() for period in periods_conn]) if ed-st >pd.Timedelta("0s")])
    for name, vals in connector_status.items()
}

In [ ]:
parts = []

for name, group in outlet_events.groupby("device_id", sort=False):
    iv = faulted_periods.get(name)
    if iv is None or len(iv) == 0:
        continue

    indexes = iv.get_indexer(group["event_time"]) != -1
    if np.any(indexes):
        parts.append(group.loc[indexes])

filtered_outlet = pd.concat(parts, ignore_index=True)

In [ ]:
total_counts_filtered = filtered_outlet.shape[0]
